In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq"
)

response = model.invoke("What is the weather in Hyderabad?")
response


AIMessage(content="I'm a large language model, I don't have real-time access to current weather conditions. But I can give you general information about Hyderabad's climate.\n\nHyderabad, the capital city of Telangana, India, has a tropical wet and dry climate. The city experiences hot summers and mild winters. Here's a breakdown of the typical weather patterns in Hyderabad:\n\n* Summer (March to May): Hot and dry, with temperatures often reaching 40°C (104°F) or more.\n* Monsoon (June to September): Warm and humid, with heavy rainfall and temperatures ranging from 25°C to 35°C (77°F to 95°F).\n* Winter (December to February): Mild and pleasant, with temperatures ranging from 15°C to 25°C (59°F to 77°F).\n* Autumn (October to November) and Spring (March to April): Warm and pleasant, with temperatures ranging from 20°C to 30°C (68°F to 86°F).\n\nTo get the current weather in Hyderabad, I recommend checking a weather website or app, such as AccuWeather, Weather.com, or the India Meteorol

In [22]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Get the current weather in a given city.
    """
    return f"The current weather in {city} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_weather])

In [23]:
response_with_tool = model_with_tools.invoke("What's the weather like in Boston?")
print(response_with_tool)
for tool_call in response_with_tool.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'z15t06h7s', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 224, 'total_tokens': 238, 'completion_time': 0.052289415, 'completion_tokens_details': None, 'prompt_time': 0.010645187, 'prompt_tokens_details': None, 'queue_time': 0.161684717, 'total_time': 0.062934602}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fd01c-6a80-73f1-82a3-f2a99baee0c3-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'z15t06h7s', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 224, 'output_tokens': 14, 'total_tokens': 238}
Tool: get_weather
Args: {'city': 'Boston'}


# Tool Execution Loop 

In [24]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

In [25]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'twpr3tj95', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 223, 'total_tokens': 237, 'completion_time': 0.053908146, 'completion_tokens_details': None, 'prompt_time': 0.020468397, 'prompt_tokens_details': None, 'queue_time': 0.052431383, 'total_time': 0.074376543}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd01c-e2be-77e3-b997-9a0b887a1cd7-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'twpr3tj95', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 14, 'total_tokens': 237}),
 ToolMessage(content='The current 